# LAB03 — Ensambles, reducción dimensional y Green AI ## Ejercicio 02 — INF-8239

## 1. Congelar el protocolo
Reutilizamos dataset, target, exclusiones y partición de test del Ejercicio 01 (LAB02).

In [2]:
##Descarga (reproducible, idéntica al Ejercicio 01):

from pathlib import Path
import sys

PROJECT_ROOT = Path.cwd().parent
sys.path.insert(0, str(PROJECT_ROOT / "src"))

from inf8239_u01.data import download_csv

URL = "https://archive.ics.uci.edu/ml/machine-learning-databases/00225/Indian%20Liver%20Patient%20Dataset%20(ILPD).csv"
destination = Path.cwd().parent / "data" / "raw" / "dataset.csv"
path = download_csv(URL, destination=str(destination))
print(path)

c:\Users\raymi\Documents\INF8239_U01\data\raw\dataset.csv


In [3]:
##Carga con nombres de columnas (igual que en el Ejercicio 01):

import pandas as pd

column_names = [
    "Age", "Gender", "TB", "DB", "Alkphos", "Sgpt", "Sgot",
    "TP", "ALB", "AG_Ratio", "Selector"
]

dataset_path = Path.cwd().parent / "data" / "raw" / "dataset.csv"
df = pd.read_csv(dataset_path, header=None, names=column_names)

print(df.shape)
assert not df.empty

(583, 11)


In [4]:
##Target, exclusiones y partición (mismos parámetros que el Ejercicio 01):

from sklearn.model_selection import train_test_split

TARGET = "Selector"

X = df.drop(columns=[TARGET])
y = df[TARGET]

Xtr, Xte, ytr, yte = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

print("Train:", Xtr.shape, " Test:", Xte.shape)

Train: (466, 10)  Test: (117, 10)


In [5]:
##Declarar métrica principal y clase prioritaria:

METRICA_PRINCIPAL = "f1_macro"
CLASE_PRIORITARIA = 1  # Enfermedad hepática — el falso negativo es el error más costoso

print(f"Métrica principal: {METRICA_PRINCIPAL}")
print(f"Clase prioritaria: {CLASE_PRIORITARIA} (enfermedad hepática)")

Métrica principal: f1_macro
Clase prioritaria: 1 (enfermedad hepática)


In [6]:
##Preprocesador (con ajuste para HistGradientBoosting):

from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler

num_cols = X.select_dtypes(include="number").columns.tolist()
cat_cols = X.select_dtypes(exclude="number").columns.tolist()

num_pipe = Pipeline([("imputer", SimpleImputer(strategy="median")),
                      ("scale", StandardScaler())])
cat_pipe = Pipeline([("imputer", SimpleImputer(strategy="most_frequent")),
                      ("onehot", OneHotEncoder(handle_unknown="ignore", sparse_output=False))])

preprocess = ColumnTransformer([("num", num_pipe, num_cols),
                                 ("cat", cat_pipe, cat_cols)])

print(len(num_cols), len(cat_cols))
print("Numéricas:", num_cols)
print("Categóricas:", cat_cols)

9 1
Numéricas: ['Age', 'TB', 'DB', 'Alkphos', 'Sgpt', 'Sgot', 'TP', 'ALB', 'AG_Ratio']
Categóricas: ['Gender']


## 2. Crear el catálogo

In [7]:
##Catálogo de 6 modelos (tal como pide el manual):

from sklearn.ensemble import RandomForestClassifier, HistGradientBoostingClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.pipeline import Pipeline
from sklearn.svm import SVC

models = {
    "logistic": Pipeline([("prep", preprocess), ("model", LogisticRegression(max_iter=2000))]),
    "svm_c1": Pipeline([("prep", preprocess), ("model", SVC(C=1, probability=True, random_state=42))]),
    "svm_c10": Pipeline([("prep", preprocess), ("model", SVC(C=10, probability=True, random_state=42))]),
    "rf_100": Pipeline([("prep", preprocess),
                         ("model", RandomForestClassifier(n_estimators=100, max_depth=8, random_state=42, n_jobs=-1))]),
    "rf_300": Pipeline([("prep", preprocess),
                         ("model", RandomForestClassifier(n_estimators=300, max_depth=8, random_state=42, n_jobs=-1))]),
    "boost": Pipeline([("prep", preprocess),
                        ("model", HistGradientBoostingClassifier(max_depth=6, random_state=42))])
}

print(list(models.keys()))

['logistic', 'svm_c1', 'svm_c10', 'rf_100', 'rf_300', 'boost']


## 3. Medir tres veces